# Custom Architecture Ablation

<a href="https://colab.research.google.com/github/openlanguagemodel/openlanguagemodel/blob/main/notebooks/03_custom_architecture_ablation_colab.ipynb" target="_blank">
  <img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open in Colab"/>
</a>

This notebook builds two tiny language models that differ only in
their block choices, then trains each for one optimizer step.

The goal is not to get a good model. The goal is to see how local an
ablation can be in OLM.

## Install OLM

In [ ]:
import importlib.util
import subprocess
import sys

if importlib.util.find_spec("olm") is None:
    subprocess.check_call([
        sys.executable,
        "-m",
        "pip",
        "install",
        "-q",
        "git+https://github.com/openlanguagemodel/openlanguagemodel.git",
    ])

## Imports And Tiny Data

In [ ]:
import random
from pathlib import Path

import torch

from olm.data.datasets import DataLoader, LocalTextDataset
from olm.data.tokenization import HFTokenizer
from olm.nn.attention import FlashAttention, GroupedQueryAttention
from olm.nn.blocks import OutputHead
from olm.nn.embeddings import Embedding
from olm.nn.feedforward import ClassicFFN, SwiGLUFFN
from olm.nn.norms import LayerNorm, RMSNorm
from olm.nn.structure import Block
from olm.nn.structure.combinators import Repeat, Residual
from olm.train import Trainer
from olm.train.optim import AdamW

seed = 7
random.seed(seed)
torch.manual_seed(seed)

device = "cuda" if torch.cuda.is_available() else "cpu"
context_length = 64
data_dir = Path("ablation_data")
data_dir.mkdir(exist_ok=True)
(data_dir / "notes.txt").write_text(
    "\n".join(
        "Attention moves information across tokens. Norms stabilize hidden states. Feed-forward layers transform features."
        for _ in range(300)
    ),
    encoding="utf-8",
)

tokenizer = HFTokenizer("gpt2")
dataset = LocalTextDataset(data_dir, tokenizer, context_length=context_length, shuffle=True, seed=seed)
loader = DataLoader(dataset, batch_size=4, num_workers=0, pin_memory=device.startswith("cuda"))
fixed_batch = next(iter(loader))
print(tuple(fixed_batch[0].shape), tuple(fixed_batch[1].shape))

## Define A Swappable Block

In [ ]:
def make_attention(kind, embed_dim, num_heads, max_seq_len, dropout):
    if kind == "mha":
        return FlashAttention(embed_dim, num_heads, dropout=dropout, causal=True)
    if kind == "gqa":
        return GroupedQueryAttention(
            embed_dim=embed_dim,
            num_heads=num_heads,
            num_kv_heads=max(1, num_heads // 2),
            max_seq_len=max_seq_len,
            dropout=dropout,
            use_bias=False,
        )
    raise ValueError(f"unknown attention kind: {kind}")


def make_ffn(kind, embed_dim, hidden_dim, dropout):
    if kind == "classic":
        return ClassicFFN(embed_dim, hidden_dim=hidden_dim, dropout=dropout)
    if kind == "swiglu":
        return SwiGLUFFN(embed_dim, hidden_dim=hidden_dim, dropout=dropout, bias=False)
    raise ValueError(f"unknown ffn kind: {kind}")


def make_norm(kind, embed_dim):
    if kind == "layernorm":
        return LayerNorm(embed_dim)
    if kind == "rmsnorm":
        return RMSNorm(embed_dim)
    raise ValueError(f"unknown norm kind: {kind}")


class AblationBlock(Block):
    def __init__(
        self,
        embed_dim,
        num_heads,
        max_seq_len,
        attention="mha",
        norm="layernorm",
        ffn="classic",
        hidden_dim=None,
        dropout=0.0,
    ):
        hidden_dim = hidden_dim or 4 * embed_dim
        super().__init__([
            Residual(Block([
                make_norm(norm, embed_dim),
                make_attention(attention, embed_dim, num_heads, max_seq_len, dropout),
            ])),
            Residual(Block([
                make_norm(norm, embed_dim),
                make_ffn(ffn, embed_dim, hidden_dim, dropout),
            ])),
        ])


class TinyAblationLM(Block):
    def __init__(
        self,
        vocab_size,
        embed_dim=128,
        num_heads=4,
        num_layers=2,
        max_seq_len=64,
        attention="mha",
        norm="layernorm",
        ffn="classic",
    ):
        embedding = Embedding(vocab_size, embed_dim)
        super().__init__([
            embedding,
            Repeat(
                lambda: AblationBlock(
                    embed_dim,
                    num_heads,
                    max_seq_len,
                    attention=attention,
                    norm=norm,
                    ffn=ffn,
                    hidden_dim=4 * embed_dim,
                ),
                num_layers,
            ),
            make_norm(norm, embed_dim),
            OutputHead(embed_dim, vocab_size, tied_embedding=embedding),
        ])

## One-Step Ablation Runner

In [ ]:
class OneBatchLoader:
    def __iter__(self):
        yield fixed_batch

    def __len__(self):
        return 1


def run_one_step(name, **model_kwargs):
    torch.manual_seed(seed)
    model = TinyAblationLM(tokenizer.vocab_size, max_seq_len=context_length, **model_kwargs)
    params = sum(p.numel() for p in model.parameters())

    trainer = Trainer(
        model,
        AdamW,
        OneBatchLoader(),
        device=device,
        context_length=context_length,
        learning_rate=1e-3,
        weight_decay=0.1,
        use_amp=device.startswith("cuda"),
        use_warmup_cosine=False,
    )
    losses = trainer.train(epochs=1, max_steps=1, log_interval=1)
    return {"name": name, "params": params, "loss": losses[-1]}

## Compare Two Blocks

In [ ]:
results = [
    run_one_step(
        "MHA + LayerNorm + ClassicFFN",
        attention="mha",
        norm="layernorm",
        ffn="classic",
    ),
    run_one_step(
        "GQA + RMSNorm + SwiGLU",
        attention="gqa",
        norm="rmsnorm",
        ffn="swiglu",
    ),
]

for result in results:
    print(f"{result['name']}: params={result['params']:,}, one-step loss={result['loss']:.4f}")

## What To Try Next

- Keep the same training loop and change only `make_attention`.
- Add QK normalization to `GroupedQueryAttention`.
- Increase `num_layers` and run more than one step.
- Move the same custom block into a reusable model file.